# NewsScope - 03. News Classification

## Phase D: Text Classification Setup & Dataset Verification

This notebook initiates **Phase D** of the NewsScope Text Mining assignment. The objective of this phase is to evaluate and compare supervised machine learning approaches for categorising news articles into four distinct topics.

### Notebook Scope
In this initial setup stage, we establish the classification workspace and perform dataset verification:
1. **Library Imports**: Import core data science and machine learning packages.
2. **Dataset Loading**: Load the preprocessed training and testing datasets using paths relative to the project root.
3. **Dimensions & Columns**: Verify dataset shapes, column names, and data types.
4. **Missing Values Check**: Confirm zero missing or invalid values in text and label columns.
5. **Class Distribution**: Verify the distribution and balance of article categories.
6. **Category & Label Verification**: Validate that all four expected AG News categories (`World`, `Sports`, `Business`, `Sci/Tech`) and their integer labels (`0`, `1`, `2`, `3`) match specifications.
7. **Sample Inspection**: Examine sample records across each category comparing original and preprocessed text representations.
8. **Roadmap for Subsequent Tasks**: Outline the upcoming document representation methods and classification algorithms.

## 1. Import Necessary Libraries

We import standard libraries for file path handling (`os`, `pathlib`), tabular data manipulation (`pandas`, `numpy`), data visualisation (`matplotlib`, `seaborn`), and the machine learning framework (`scikit-learn`).

In [1]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn

# Display package versions for reproducibility
print("Environment & Library Versions:")
print(f"  Pandas:       {pd.__version__}")
print(f"  NumPy:        {np.__version__}")
print(f"  Scikit-Learn: {sklearn.__version__}")
print(f"  Working Dir:  {Path.cwd()}")

Environment & Library Versions:
  Pandas:       3.1.1
  NumPy:        2.5.4
  Scikit-Learn: 1.9.1
  Working Dir:  C:\Users\Acer\Downloads\NewsScope


## 2. Locate and Load Processed Datasets

We locate the project root dynamically and load `train_processed.csv` and `test_processed.csv` from `data/processed/`.
This relative path resolution works reliably whether the notebook is executed from the repository root or from the `notebooks/` directory.

In [2]:
# Resolve project root and data directory dynamically
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
processed_dir = project_root / "data" / "processed"

train_csv_path = processed_dir / "train_processed.csv"
test_csv_path = processed_dir / "test_processed.csv"

# Verify file existence before loading
if not train_csv_path.exists():
    raise FileNotFoundError(f"Training dataset not found at expected path: {train_csv_path}")
if not test_csv_path.exists():
    raise FileNotFoundError(f"Testing dataset not found at expected path: {test_csv_path}")

# Load processed datasets (preserving original datasets intact)
train_df = pd.read_csv(train_csv_path)
test_df = pd.read_csv(test_csv_path)

print("Successfully loaded processed datasets:")
print(f"  Train path: {train_csv_path}")
print(f"  Test path:  {test_csv_path}")

Successfully loaded processed datasets:
  Train path: C:\Users\Acer\Downloads\NewsScope\data\processed\train_processed.csv
  Test path:  C:\Users\Acer\Downloads\NewsScope\data\processed\test_processed.csv


## 3. Dataset Dimensions & Available Columns

We inspect the dimensions (number of rows and columns) and review the schema of both DataFrames.

In [3]:
# Display dataset dimensions
print("=== Dataset Dimensions ===")
print(f"Training set: {train_df.shape[0]:,} rows, {train_df.shape[1]} columns")
print(f"Test set:     {test_df.shape[0]:,} rows, {test_df.shape[1]} columns")

# Display available columns
print("\n=== Available Columns in Training Set ===")
for col in train_df.columns:
    sample_val = str(train_df[col].iloc[0])
    preview = sample_val[:60] + "..." if len(sample_val) > 60 else sample_val
    print(f"  - {col:<20} (dtype: {str(train_df[col].dtype):<7}) | Sample: {preview}")

# Check DataFrame information
print("\n=== Training DataFrame Info ===")
train_df.info()

=== Dataset Dimensions ===
Training set: 120,000 rows, 5 columns
Test set:     7,600 rows, 5 columns

=== Available Columns in Training Set ===
  - text                 (dtype: str    ) | Sample: Wall St. Bears Claw Back Into the Black (Reuters) Reuters - ...
  - label                (dtype: int64  ) | Sample: 2
  - category             (dtype: str    ) | Sample: Business
  - clean_text           (dtype: str    ) | Sample: wall st bear claw back black reuters reuters short seller wa...
  - clean_text_stemmed   (dtype: str    ) | Sample: wall st bear claw back black reuter reuter short seller wall...

=== Training DataFrame Info ===
<class 'pandas.DataFrame'>
RangeIndex: 120000 entries, 0 to 119999
Data columns (total 5 columns):
 #   Column              Non-Null Count   Dtype
---  ------              --------------   -----
 0   text                120000 non-null  str  
 1   label               120000 non-null  int64
 2   category            120000 non-null  str  
 3   clean_text      

## 4. Missing Values Analysis

Data integrity is critical for training supervised models. We verify that there are no missing (`NaN` or `None`) values across all columns, and also confirm there are no empty strings in the text columns.

In [4]:
# Check missing values per column in both datasets
missing_summary = pd.DataFrame({
    "Train Missing": train_df.isna().sum(),
    "Test Missing": test_df.isna().sum()
})
print("=== Missing Values Count ===")
display(missing_summary)

# Check for empty string entries in text columns
empty_text_train = (train_df["clean_text"].astype(str).str.strip() == "").sum()
empty_text_test = (test_df["clean_text"].astype(str).str.strip() == "").sum()

print(f"\nEmpty 'clean_text' entries in train: {empty_text_train}")
print(f"Empty 'clean_text' entries in test:  {empty_text_test}")

# Assert data completeness
assert train_df.isna().sum().sum() == 0, "Found missing values in training set!"
assert test_df.isna().sum().sum() == 0, "Found missing values in test set!"
print("\nData integrity confirmed: 0 missing or empty values detected.")

=== Missing Values Count ===

Empty 'clean_text' entries in train: 0
Empty 'clean_text' entries in test:  0

Data integrity confirmed: 0 missing or empty values detected.


,Train Missing,Test Missing
text,0,0
label,0,0
category,0,0
clean_text,0,0
clean_text_stemmed,0,0


## 5. Class Distribution

We inspect the distribution of target classes across both training and testing datasets to verify class balance.

In [5]:
# Compute count and proportion of each category
train_dist = train_df["category"].value_counts()
test_dist = test_df["category"].value_counts()

dist_comparison = pd.DataFrame({
    "Train Count": train_dist,
    "Train %": (train_dist / len(train_df) * 100).round(2),
    "Test Count": test_dist,
    "Test %": (test_dist / len(test_df) * 100).round(2)
})

print("=== Class Distribution Across Splits ===")
display(dist_comparison)

# Confirm class balance
is_train_balanced = len(train_dist.unique()) == 1
is_test_balanced = len(test_dist.unique()) == 1
print(f"Training set is perfectly balanced: {is_train_balanced} ({train_dist.iloc[0]:,} per class)")
print(f"Test set is perfectly balanced:     {is_test_balanced} ({test_dist.iloc[0]:,} per class)")

=== Class Distribution Across Splits ===
Training set is perfectly balanced: True (30,000 per class)
Test set is perfectly balanced:     True (1,900 per class)


,Train Count,Train %,Test Count,Test %
category,,,,
Business,30000,25.0,1900,25.0
Sci/Tech,30000,25.0,1900,25.0
Sports,30000,25.0,1900,25.0
World,30000,25.0,1900,25.0


## 6. Category & Label Verification

We confirm that the training and testing datasets contain exactly the expected four AG News categories:
- `0`: **World**
- `1`: **Sports**
- `2`: **Business**
- `3`: **Sci/Tech**

We also verify that the numerical `label` and string `category` columns are consistently aligned across all records.

In [6]:
# Expected AG News category specifications
expected_mapping = {
    0: "World",
    1: "Sports",
    2: "Business",
    3: "Sci/Tech"
}
expected_categories = set(expected_mapping.values())
expected_labels = set(expected_mapping.keys())

# Extract actual categories and labels present in datasets
train_categories = set(train_df["category"].unique())
test_categories = set(test_df["category"].unique())
train_labels = set(train_df["label"].unique())
test_labels = set(test_df["label"].unique())

print("=== Category & Label Verification ===")
print(f"Expected categories: {sorted(expected_categories)}")
print(f"Train categories:    {sorted(train_categories)}")
print(f"Test categories:     {sorted(test_categories)}")

# Verify matching categories and labels
assert train_categories == expected_categories, "Train categories do not match expected categories!"
assert test_categories == expected_categories, "Test categories do not match expected categories!"
assert train_labels == expected_labels, "Train labels do not match expected labels!"
assert test_labels == expected_labels, "Test labels do not match expected labels!"

# Verify mapping consistency for every single row
train_mapping_consistent = (train_df["label"].map(expected_mapping) == train_df["category"]).all()
test_mapping_consistent = (test_df["label"].map(expected_mapping) == test_df["category"]).all()

print(f"Train label-category mapping 100% consistent: {train_mapping_consistent}")
print(f"Test label-category mapping 100% consistent:  {test_mapping_consistent}")
print("\nCategory and label integrity confirmed for all four AG News classes.")

=== Category & Label Verification ===
Expected categories: ['Business', 'Sci/Tech', 'Sports', 'World']
Train categories:    ['Business', 'Sci/Tech', 'Sports', 'World']
Test categories:     ['Business', 'Sci/Tech', 'Sports', 'World']
Train label-category mapping 100% consistent: True
Test label-category mapping 100% consistent:  True

Category and label integrity confirmed for all four AG News classes.


## 7. Sample Records Inspection

We inspect representative samples from each of the four categories to observe the preprocessed representations (`clean_text` lemmatized, `clean_text_stemmed`) alongside the original article `text`.

In [7]:
# Display one sample from each category
samples = train_df.drop_duplicates(subset=["category"]).sort_values("label").reset_index(drop=True)

print("=== Sample Records Across the Four Categories ===\n")
for idx, row in samples.iterrows():
    print(f"--- Category: {row['category']} (Label {row['label']}) ---")
    print(f"  Original Text:       {row['text'][:120]}...")
    print(f"  Clean (Lemmatized):  {row['clean_text'][:120]}...")
    print(f"  Clean (Stemmed):     {row['clean_text_stemmed'][:120]}...\n")

# Display tabular preview
display(train_df[["label", "category", "clean_text"]].head(5))

=== Sample Records Across the Four Categories ===

--- Category: World (Label 0) ---
  Original Text:       Venezuelans Vote Early in Referendum on Chavez Rule (Reuters) Reuters - Venezuelans turned out early\and in large number...
  Clean (Lemmatized):  venezuelan vote early referendum chavez rule reuters reuters venezuelan turned early large number sunday vote historic r...
  Clean (Stemmed):     venezuelan vote earli referendum chavez rule reuter reuter venezuelan turn earli larg number sunday vote histor referend...

--- Category: Sports (Label 1) ---
  Original Text:       Phelps, Thorpe Advance in 200 Freestyle (AP) AP - Michael Phelps took care of qualifying for the Olympic 200-meter frees...
  Clean (Lemmatized):  phelps thorpe advance freestyle ap ap michael phelps took care qualifying olympic meter freestyle semifinal sunday found...
  Clean (Stemmed):     phelp thorp advanc freestyl ap ap michael phelp took care qualifi olymp meter freestyl semifin sunday found ad american .

,label,category,clean_text
0,2,Business,wall st bear claw back black reuters reuters s...
1,2,Business,carlyle look toward commercial aerospace reute...
2,2,Business,oil economy cloud stock outlook reuters reuter...
3,2,Business,iraq halt oil export main southern pipeline re...
4,2,Business,oil price soar time record posing new menace u...


## 8. Summary & Roadmap for Phase D Classification

### Setup Verification Summary
- **Training Dataset**: 120,000 articles, perfectly balanced across 4 categories (30,000 articles per category).
- **Testing Dataset**: 7,600 articles, perfectly balanced across 4 categories (1,900 articles per category).
- **Target Classes**: `World` (0), `Sports` (1), `Business` (2), and `Sci/Tech` (3).
- **Data Integrity**: 0 missing values, 0 empty strings, 100% consistent label-to-category mapping.
- **Text Features**: Both lemmatized (`clean_text`) and stemmed (`clean_text_stemmed`) representations are ready for vectorisation.

### Planned Next Tasks
In subsequent tasks, we will implement and compare:

1. **Document Representation Methods** (at least three):
   - **Bag-of-Words (CountVectorizer)**: Unigram term-frequency baseline.
   - **TF-IDF Vectorisation (TfidfVectorizer)**: Term Frequency-Inverse Document Frequency weighting (unigrams and unigrams + bigrams).
   - **Word Embeddings or N-gram representations**: Dense representation or character/subword n-grams.

2. **Classification Algorithms** (at least three):
   - **Multinomial Naive Bayes (MNB)**: Fast probabilistic benchmark ideal for discrete word count/TF-IDF distributions.
   - **Logistic Regression (Softmax)**: Linear discriminative model with calibrated probability outputs.
   - **Linear Support Vector Classifier (LinearSVC)**: Maximum-margin classifier well-suited for high-dimensional sparse text vectors.

3. **Comprehensive Evaluation**:
   - Accuracy, Precision, Recall, Macro-F1 score, and Confusion Matrices on the held-out test set.